In [143]:
import pandas as pd 
import numpy as np

In [144]:
comment_counter=pd.read_csv('comment_counter.csv')
Succeful_sub=pd.read_csv("Succeful_sub.csv")
submission_counter=pd.read_csv("submission_counter.csv")
discussions_counter=pd.read_csv("discussions_counter.csv")
test=pd.read_csv('test.csv')
train=pd.read_csv('train.csv')

In [145]:
comment_counter.drop('UserMonthYear', axis = 1, inplace = True)
Succeful_sub.drop('UserMonthYear', axis = 1, inplace = True)
submission_counter.drop('UserMonthYear', axis = 1, inplace = True)
discussions_counter.drop('UserMonthYear', axis = 1, inplace = True)

In [146]:
test

,User_ID,month,year
0,ID_H1ELY25E,1,4
1,ID_H1ELY25E,2,4
2,ID_H1ELY25E,3,4
3,ID_463Q2BCO,1,4
4,ID_463Q2BCO,2,4
...,...,...,...
65218,ID_4XKWR8UN,3,4
65219,ID_L54061S5,3,4
65220,ID_I3C1N5RO,3,4
65221,ID_WWNR6I15,3,4


on a fait la concaténation des 3 colonnes User, month et year 

In [147]:
test['UserMonthYear'] = test['User_ID'].astype(str)+'_'+test['month'].astype(str)+'_'+test['year'].astype(str)
test

,User_ID,month,year,UserMonthYear
0,ID_H1ELY25E,1,4,ID_H1ELY25E_1_4
1,ID_H1ELY25E,2,4,ID_H1ELY25E_2_4
2,ID_H1ELY25E,3,4,ID_H1ELY25E_3_4
3,ID_463Q2BCO,1,4,ID_463Q2BCO_1_4
4,ID_463Q2BCO,2,4,ID_463Q2BCO_2_4
...,...,...,...,...
65218,ID_4XKWR8UN,3,4,ID_4XKWR8UN_3_4
65219,ID_L54061S5,3,4,ID_L54061S5_3_4
65220,ID_I3C1N5RO,3,4,ID_I3C1N5RO_3_4
65221,ID_WWNR6I15,3,4,ID_WWNR6I15_3_4


# Merge

on a ajouté à test les colonnes des autres datas dans le but de calculer le score et la prédiction 

In [148]:
test = test.merge(discussions_counter, on='User_ID')

In [149]:
test = test.merge(submission_counter, on='User_ID')
test = test.merge(Succeful_sub, on='User_ID')
test = test.merge(comment_counter, on='User_ID')

In [150]:
test.drop_duplicates(inplace = True)

In [151]:
test

,User_ID,month,year,UserMonthYear,discussions_counter,submission_counter,PublicRank,comment_counter
0,ID_IYC1NOTL,1,4,ID_IYC1NOTL_1_4,1,0,3,4
1,ID_IYC1NOTL,2,4,ID_IYC1NOTL_2_4,1,0,3,4
2,ID_IYC1NOTL,3,4,ID_IYC1NOTL_3_4,1,0,3,4
3,ID_F9RXMPBT,1,4,ID_F9RXMPBT_1_4,1,0,11,4
4,ID_F9RXMPBT,1,4,ID_F9RXMPBT_1_4,1,0,0,4
...,...,...,...,...,...,...,...,...
35388972,ID_KK440U4N,3,4,ID_KK440U4N_3_4,1,0,2,9
35388976,ID_KK440U4N,3,4,ID_KK440U4N_3_4,1,1,11,9
35388978,ID_KK440U4N,3,4,ID_KK440U4N_3_4,1,1,0,9
35388986,ID_KK440U4N,3,4,ID_KK440U4N_3_4,1,1,1,9


# Train

In [152]:
train

,User_ID,month,year,CompPart,Comment,Sub,Disc,Target
0,ID_XI7BAR4Y,8,3,0,0,0,0,0
1,ID_XI7BAR4Y,8,2,0,0,0,0,0
2,ID_XI7BAR4Y,9,2,0,0,0,0,0
3,ID_XI7BAR4Y,9,3,0,0,0,0,0
4,ID_XI7BAR4Y,10,3,0,0,0,0,0
...,...,...,...,...,...,...,...,...
259827,ID_MAP5X6D4,12,3,0,0,0,0,0
259828,ID_QHUAHU76,12,3,0,0,0,0,0
259829,ID_8IKU22O5,12,3,1,0,0,0,1
259830,ID_NHWCR1IY,12,3,0,0,0,0,0


In [153]:
y=train['Target']

In [154]:
x=train[['month','year']]

In [155]:
x_CompPart=train[['month','year','CompPart']]

In [156]:
x_Comment=train[['month','year','CompPart','Comment']]

In [157]:
x_Sub=train[['month','year','CompPart','Comment','Sub']]

In [158]:
x_Disc=train[['month','year','CompPart','Comment','Sub','Disc']]

# Train Score en utilisant knn

In [159]:
from sklearn.neighbors import KNeighborsClassifier

In [160]:
model = KNeighborsClassifier()

In [161]:
x1=x_Disc.copy()

In [162]:
model.fit(x1,y)
model.score(x1,y)

0.9999769081560392

# Test Model

In [163]:
del test['User_ID']

In [164]:
test=test.drop('UserMonthYear',axis=1)

In [165]:
test

,month,year,discussions_counter,submission_counter,PublicRank,comment_counter
0,1,4,1,0,3,4
1,2,4,1,0,3,4
2,3,4,1,0,3,4
3,1,4,1,0,11,4
4,1,4,1,0,0,4
...,...,...,...,...,...,...
35388972,3,4,1,0,2,9
35388976,3,4,1,1,11,9
35388978,3,4,1,1,0,9
35388986,3,4,1,1,1,9


In [166]:
y_pred=model.predict(test)

In [167]:
predict=pd.DataFrame(model.predict(test))
predict

,0
0,1
1,1
2,1
3,1
4,1
...,...
89422,1
89423,1
89424,1
89425,1


In [168]:
test1=pd.read_csv('test.csv')
test1['UserMonthYear'] = test1['User_ID'].astype(str)+'_'+test1['month'].astype(str)+'_'+test1['year'].astype(str)
test1

,User_ID,month,year,UserMonthYear
0,ID_H1ELY25E,1,4,ID_H1ELY25E_1_4
1,ID_H1ELY25E,2,4,ID_H1ELY25E_2_4
2,ID_H1ELY25E,3,4,ID_H1ELY25E_3_4
3,ID_463Q2BCO,1,4,ID_463Q2BCO_1_4
4,ID_463Q2BCO,2,4,ID_463Q2BCO_2_4
...,...,...,...,...
65218,ID_4XKWR8UN,3,4,ID_4XKWR8UN_3_4
65219,ID_L54061S5,3,4,ID_L54061S5_3_4
65220,ID_I3C1N5RO,3,4,ID_I3C1N5RO_3_4
65221,ID_WWNR6I15,3,4,ID_WWNR6I15_3_4


In [169]:
idd=pd.DataFrame(test1['UserMonthYear'])

In [170]:
predict['UserMonthYear']=idd

In [171]:
predict.columns=['Target','UserMonthYear']

In [172]:
predict = np.fliplr(predict)

In [173]:
predict=pd.DataFrame(predict)

In [174]:
predict.columns=['UserMonthYear','Target']

In [175]:
predict

,UserMonthYear,Target
0,ID_H1ELY25E_1_4,1
1,ID_H1ELY25E_2_4,1
2,ID_H1ELY25E_3_4,1
3,ID_463Q2BCO_1_4,1
4,ID_463Q2BCO_2_4,1
...,...,...
89422,NaN,1
89423,NaN,1
89424,NaN,1
89425,NaN,1


In [176]:
predict.dropna(subset = ["UserMonthYear"], inplace=True)
predict

,UserMonthYear,Target
0,ID_H1ELY25E_1_4,1
1,ID_H1ELY25E_2_4,1
2,ID_H1ELY25E_3_4,1
3,ID_463Q2BCO_1_4,1
4,ID_463Q2BCO_2_4,1
...,...,...
65218,ID_4XKWR8UN_3_4,1
65219,ID_L54061S5_3_4,1
65220,ID_I3C1N5RO_3_4,1
65221,ID_WWNR6I15_3_4,1


In [177]:
predict.to_csv("PredictV2.csv", index = False)

# test de decision tree

nous utilisons la methode decision tree de SK-learn pour mieux visualiser la prediction et la correlation entre les colonnes 

train_test_split : cette fonction divise notre data train en 2 datas train et test : 70% train et 30% test. ("juste pour le tester ") 

In [178]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(x_Disc, y, test_size=0.344153, random_state=0)

In [179]:
print(X_train)

        month  year  CompPart  Comment  Sub  Disc
167936     11     3         0        0    0     0
76889       6     3         0        0    0     0
92052       7     3         0        0    0     0
248961      6     2         0        0    0     0
256421     10     3         1        0    1     0
...       ...   ...       ...      ...  ...   ...
176963     12     1         0        0    0     0
117952      2     3         0        0    0     0
173685      7     3         0        0    0     0
43567       9     2         0        0    0     0
199340      3     2         1        0    1     0

[170410 rows x 6 columns]


In [180]:
from sklearn.tree import DecisionTreeClassifier

In [181]:
tree = DecisionTreeClassifier()

In [182]:
fit=tree.fit(X_train, y_train)

In [183]:
y_pred=tree.predict(X_test)

In [184]:
tree_predict=pd.DataFrame(tree.predict(X_test))
tree_predict

,0
0,0
1,0
2,0
3,0
4,0
...,...
89417,0
89418,0
89419,1
89420,0


In [185]:
idd

,UserMonthYear
0,ID_H1ELY25E_1_4
1,ID_H1ELY25E_2_4
2,ID_H1ELY25E_3_4
3,ID_463Q2BCO_1_4
4,ID_463Q2BCO_2_4
...,...
65218,ID_4XKWR8UN_3_4
65219,ID_L54061S5_3_4
65220,ID_I3C1N5RO_3_4
65221,ID_WWNR6I15_3_4


In [186]:
tree_predict['UserMonthYear']=idd

tree_predict.columns=['Target','UserMonthYear']
tree_predict

,Target,UserMonthYear
0,0,ID_H1ELY25E_1_4
1,0,ID_H1ELY25E_2_4
2,0,ID_H1ELY25E_3_4
3,0,ID_463Q2BCO_1_4
4,0,ID_463Q2BCO_2_4
...,...,...
89417,0,NaN
89418,0,NaN
89419,1,NaN
89420,0,NaN


In [187]:
tree_predict.columns=['Target','UserMonthYear']

tree_predict = np.fliplr(tree_predict)

tree_predict=pd.DataFrame(tree_predict)

tree_predict.columns=['UserMonthYear','Target']

tree_predict

,UserMonthYear,Target
0,ID_H1ELY25E_1_4,0
1,ID_H1ELY25E_2_4,0
2,ID_H1ELY25E_3_4,0
3,ID_463Q2BCO_1_4,0
4,ID_463Q2BCO_2_4,0
...,...,...
89417,NaN,0
89418,NaN,0
89419,NaN,1
89420,NaN,0


In [188]:
tree_predict.dropna(subset = ["UserMonthYear"], inplace=True)
tree_predict

tree_predict.to_csv("TreePredict.csv", index = False)

# test tree decision 

par contre ici on travaille sur test et train donné et notre propres colonnes 

In [189]:
fite=tree.fit(x_Disc, y)
fite

DecisionTreeClassifier()

In [190]:
test_pred=tree.predict(test)
test_pred

array([1, 1, 1, ..., 1, 1, 1], dtype=int64)

In [191]:
test_pred=pd.DataFrame(test_pred)
test_pred

,0
0,1
1,1
2,1
3,1
4,1
...,...
89422,1
89423,1
89424,1
89425,1


In [192]:
test_pred.value_counts()

1    89427
dtype: int64

In [193]:
test_pred['UserMonthYear']=idd

test_pred.columns=['Target','UserMonthYear']
test_pred

test_pred.columns=['Target','UserMonthYear']

test_pred = np.fliplr(test_pred)

test_pred=pd.DataFrame(test_pred)

test_pred.columns=['UserMonthYear','Target']



test_pred.dropna(subset = ["UserMonthYear"], inplace=True)


test_pred.to_csv("TreetestPred.csv", index = False)

ici on va a chaque fois ajouter une seule colonne de train  

## month year

In [194]:
x=train[['month','year']]

In [195]:
test_my=test[['month','year']]

In [196]:
m_y=tree.fit(x, y)
m_y

DecisionTreeClassifier()

In [197]:
test_pred_my=tree.predict(test_my)
test_pred_my

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [198]:
test_pred_my=pd.DataFrame(test_pred_my)
test_pred_my

,0
0,0
1,0
2,0
3,0
4,0
...,...
89422,0
89423,0
89424,0
89425,0


In [199]:
test_pred_my.value_counts()

0    89427
dtype: int64

In [200]:
test_pred_my['UserMonthYear']=idd

test_pred_my.columns=['Target','UserMonthYear']

test_pred_my.columns=['Target','UserMonthYear']

test_pred_my = np.fliplr(test_pred_my)

test_pred_my=pd.DataFrame(test_pred_my)

test_pred_my.columns=['UserMonthYear','Target']



test_pred_my.dropna(subset = ["UserMonthYear"], inplace=True)


test_pred_my.to_csv("TreetestPredmonthyear.csv", index = False)

## month year compart 

In [201]:
x_CompPart=train[['month','year','CompPart']]

In [202]:
y

0         0
1         0
2         0
3         0
4         0
         ..
259827    0
259828    0
259829    1
259830    0
259831    1
Name: Target, Length: 259832, dtype: int64

In [203]:
test_comp=test[['month','year','PublicRank']]

In [204]:
m_y_comp=tree.fit(x_CompPart, y)
m_y_comp

DecisionTreeClassifier()

In [205]:
test_pred_my_comp=tree.predict(test_comp)
test_pred_my_comp

array([1, 1, 1, ..., 0, 1, 1], dtype=int64)

In [206]:
test_pred_my_comp=pd.DataFrame(test_pred_my_comp)
test_pred_my_comp

,0
0,1
1,1
2,1
3,1
4,0
...,...
89422,1
89423,1
89424,0
89425,1


In [207]:
test_pred_my_comp.value_counts()

1    69759
0    19668
dtype: int64

In [208]:
test_pred_my_comp['UserMonthYear']=idd

test_pred_my_comp.columns=['Target','UserMonthYear']

test_pred_my_comp.columns=['Target','UserMonthYear']

test_pred_my_comp = np.fliplr(test_pred_my_comp)

test_pred_my_comp=pd.DataFrame(test_pred_my_comp)

test_pred_my_comp.columns=['UserMonthYear','Target']



test_pred_my_comp.dropna(subset = ["UserMonthYear"], inplace=True)


test_pred_my_comp.to_csv("TreetestPredMonthYearComp.csv", index = False)

## month year comp comment

In [209]:
x_Comment=train[['month','year','CompPart','Comment']]

In [210]:
test_comment=test[['month','year','PublicRank','comment_counter']]

In [211]:
m_y_comment=tree.fit(x_Comment, y)
m_y_comment

DecisionTreeClassifier()

In [212]:
test_pred_my_comment=tree.predict(test_comment)
test_pred_my_comment

array([1, 1, 1, ..., 1, 1, 1], dtype=int64)

In [213]:
test_pred_my_comment=pd.DataFrame(test_pred_my_comment)
test_pred_my_comment

,0
0,1
1,1
2,1
3,1
4,1
...,...
89422,1
89423,1
89424,1
89425,1


In [214]:
test_pred_my_comment.value_counts()

1    89427
dtype: int64

In [215]:
test_pred_my_comment['UserMonthYear']=idd

test_pred_my_comment.columns=['Target','UserMonthYear']

test_pred_my_comment.columns=['Target','UserMonthYear']

test_pred_my_comment = np.fliplr(test_pred_my_comment)

test_pred_my_comment=pd.DataFrame(test_pred_my_comment)

test_pred_my_comment.columns=['UserMonthYear','Target']



test_pred_my_comment.dropna(subset = ["UserMonthYear"], inplace=True)


test_pred_my_comment.to_csv("TreetestPredMonthYearCompComment.csv", index = False)

## month year comp comment submission

In [216]:
x_Sub=train[['month','year','CompPart','Comment','Sub']]

In [217]:
test_sub=test[['month','year','PublicRank','comment_counter','submission_counter']]

In [218]:
m_y_sub=tree.fit(x_Sub, y)
m_y_sub

DecisionTreeClassifier()

In [219]:
test_pred_sub=tree.predict(test_sub)
test_pred_sub

array([1, 1, 1, ..., 1, 1, 1], dtype=int64)

In [220]:
test_pred_sub=pd.DataFrame(test_pred_sub)
test_pred_sub

,0
0,1
1,1
2,1
3,1
4,1
...,...
89422,1
89423,1
89424,1
89425,1


In [221]:
test_pred_sub.value_counts()

1    89427
dtype: int64

In [222]:
test_pred_sub['UserMonthYear']=idd

test_pred_sub.columns=['Target','UserMonthYear']

test_pred_sub.columns=['Target','UserMonthYear']

test_pred_sub = np.fliplr(test_pred_sub)

test_pred_sub=pd.DataFrame(test_pred_sub)

test_pred_sub.columns=['UserMonthYear','Target']

test_pred_sub.dropna(subset = ["UserMonthYear"], inplace=True)


test_pred_sub.to_csv("TreetestPredMonthYearCompCommentSub.csv", index = False)

# LogisticRegression

In [223]:
from sklearn.linear_model import LogisticRegression

In [224]:
log = LogisticRegression()

In [225]:
log.fit(x_Disc, y)
log.score(x_Disc,y)

1.0

In [226]:
log_pred=log.predict(test)
log_pred

array([1, 1, 1, ..., 1, 1, 1], dtype=int64)

In [227]:
log_pred=pd.DataFrame(log_pred)
log_pred

,0
0,1
1,1
2,1
3,1
4,1
...,...
89422,1
89423,1
89424,1
89425,1


In [228]:
log_pred.value_counts()

1    89427
dtype: int64

In [229]:
log_pred['UserMonthYear']=idd

log_pred.columns=['Target','UserMonthYear']

log_pred.columns=['Target','UserMonthYear']

log_pred = np.fliplr(log_pred)

log_pred=pd.DataFrame(log_pred)

log_pred.columns=['UserMonthYear','Target']

log_pred.dropna(subset = ["UserMonthYear"], inplace=True)


log_pred.to_csv("logPred.csv", index = False)

# month year log 

In [230]:
x=train[['month','year']]

In [231]:
test_my=test[['month','year']]

In [232]:
log_my=log.fit(x,y)
log_my
log.score(x,y)

0.8602712521937251

In [233]:
log_pred_my=log.predict(test_my)
log_pred_my

array([0, 0, 0, ..., 0, 0, 0], dtype=int64)

In [234]:
log_pred_my=pd.DataFrame(log_pred_my)
log_pred_my

,0
0,0
1,0
2,0
3,0
4,0
...,...
89422,0
89423,0
89424,0
89425,0


In [235]:
log_pred_my.value_counts()

0    89427
dtype: int64

In [236]:
log_pred_my['UserMonthYear']=idd

log_pred_my.columns=['Target','UserMonthYear']

log_pred_my.columns=['Target','UserMonthYear']

log_pred_my = np.fliplr(log_pred_my)

log_pred_my=pd.DataFrame(log_pred_my)

log_pred_my.columns=['UserMonthYear','Target']

log_pred_my.dropna(subset = ["UserMonthYear"], inplace=True)

log_pred_my.to_csv("logPredMonthYear.csv", index = False)

# month year comppart

In [237]:
x_CompPart=train[['month','year','CompPart']]

In [238]:
test_comp=test[['month','year','PublicRank']]

In [239]:
log_comp=log.fit(x_CompPart,y)
log_comp
log.score(x_CompPart,y)

0.9811378121247575

In [240]:
log_pred_comp=pd.DataFrame(log.predict(test_comp))
log_pred_comp

,0
0,1
1,1
2,1
3,1
4,0
...,...
89422,1
89423,1
89424,0
89425,1


In [241]:
log_pred_comp.value_counts()

1    69759
0    19668
dtype: int64

In [242]:
log_pred_comp['UserMonthYear']=idd

log_pred_comp.columns=['Target','UserMonthYear']

log_pred_comp.columns=['Target','UserMonthYear']

log_pred_comp = np.fliplr(log_pred_comp)

log_pred_comp=pd.DataFrame(log_pred_comp)

log_pred_comp.columns=['UserMonthYear','Target']

log_pred_comp.dropna(subset = ["UserMonthYear"], inplace=True)


log_pred_comp.to_csv("logPredMonthYearComp.csv", index = False)

# month year comp comment

In [243]:
x_Comment=train[['month','year','CompPart','Comment']]

In [244]:
test_comment=test[['month','year','PublicRank','comment_counter']]

In [245]:
log_comment=log.fit(x_Comment,y)
log_comment
log.score(x_Comment,y)

0.9852789494750455

In [246]:
log_pred_comment=pd.DataFrame(log.predict(test_comment))
log_pred_comment

,0
0,1
1,1
2,1
3,1
4,1
...,...
89422,1
89423,1
89424,1
89425,1


In [247]:
log_pred_comment.value_counts()

1    89427
dtype: int64

In [248]:
log_pred_comment['UserMonthYear']=idd

log_pred_comment.columns=['Target','UserMonthYear']

log_pred_comment.columns=['Target','UserMonthYear']

log_pred_comment = np.fliplr(log_pred_comment)

log_pred_comment=pd.DataFrame(log_pred_comment)

log_pred_comment.columns=['UserMonthYear','Target']

log_pred_comment.dropna(subset = ["UserMonthYear"], inplace=True)

log_pred_comment.to_csv("logPredMonthYearCompComment.csv", index = False)

# month year comp comment submission

In [249]:
x_Sub=train[['month','year','CompPart','Comment','Sub']]

In [250]:
test_sub=test[['month','year','PublicRank','comment_counter','submission_counter']]

In [251]:
log_sub=log.fit(x_Sub,y)
log_sub
log.score(x_Sub,y)

0.9981834416084239

In [252]:
log_pred_sub=pd.DataFrame(log.predict(test_sub))
log_pred_sub

,0
0,1
1,1
2,1
3,1
4,1
...,...
89422,1
89423,1
89424,1
89425,1


In [253]:
log_pred_sub.value_counts()

1    89427
dtype: int64

In [254]:
log_pred_sub['UserMonthYear']=idd

log_pred_sub.columns=['Target','UserMonthYear']

log_pred_sub.columns=['Target','UserMonthYear']

log_pred_sub = np.fliplr(log_pred_sub)

log_pred_sub=pd.DataFrame(log_pred_sub)

log_pred_sub.columns=['UserMonthYear','Target']

log_pred_sub.dropna(subset = ["UserMonthYear"], inplace=True)


log_pred_sub.to_csv("logPredMonthYearCompCommentSub.csv", index = False)

# GaussianNB

In [255]:
from sklearn.naive_bayes import GaussianNB


In [256]:
guass = GaussianNB()
guass.fit(x_Disc, y)
guass.score(x_Disc,y)

1.0

In [257]:
guass_pred = pd.DataFrame(guass.predict(test))
guass_pred

,0
0,1
1,1
2,1
3,1
4,1
...,...
89422,1
89423,1
89424,1
89425,1


In [258]:
guass_pred.value_counts()

1    89427
dtype: int64

In [259]:
guass_pred['UserMonthYear']=idd

guass_pred.columns=['Target','UserMonthYear']

guass_pred.columns=['Target','UserMonthYear']

guass_pred = np.fliplr(guass_pred)

guass_pred=pd.DataFrame(guass_pred)

guass_pred.columns=['UserMonthYear','Target']

guass_pred.dropna(subset = ["UserMonthYear"], inplace=True)

guass_pred.to_csv("guassPred.csv", index = False)

# linear regression 

In [260]:
from sklearn.linear_model import LinearRegression


In [261]:
reg = LinearRegression()

In [262]:
model.fit(x_Disc,y)
model.score(x_Disc,y)

0.9999769081560392

In [263]:
reg_pred=model.predict(test)
reg_pred

array([1, 1, 1, ..., 1, 1, 1], dtype=int64)

In [264]:
reg_predd=pd.DataFrame(model.predict(test))

In [265]:
reg_predd.to_csv("regPred.csv", index = False)

In [266]:
for i in range(len(reg_pred)):
    if reg_pred[i] >= 5:
        reg_pred[i] = 1
    else:
        reg_pred[i] = 0

In [267]:
reg_pred1=pd.DataFrame(reg_pred)
reg_pred1.astype(int)

,0
0,0
1,0
2,0
3,0
4,0
...,...
89422,0
89423,0
89424,0
89425,0


In [268]:
reg_predd.value_counts()

1    89427
dtype: int64

In [269]:
reg_pred1['UserMonthYear']=idd

reg_pred1.columns=['Target','UserMonthYear']

reg_pred1.columns=['Target','UserMonthYear']

reg_pred1 = np.fliplr(reg_pred1)

reg_pred1=pd.DataFrame(reg_pred1)

reg_pred1.columns=['UserMonthYear','Target']

reg_pred1.dropna(subset = ["UserMonthYear"], inplace=True)

reg_pred1.to_csv("regPred1.csv", index = False)

# SVM

In [ ]:
from sklearn.svm import SVR

In [ ]:
model = SVR(C=100)

In [ ]:
model.fit(x_Disc,y)
model.score(x_Disc,y)

In [ ]:
svm_pred=pd.DataFrame(model.predict(test))

In [ ]:
svm_pred['UserMonthYear']=idd

svm_pred.columns=['Target','UserMonthYear']

svm_pred.columns=['Target','UserMonthYear']

svm_pred = np.fliplr(svm_pred)

svm_pred=pd.DataFrame(svm_pred)

svm_pred.columns=['UserMonthYear','Target']

svm_pred.dropna(subset = ["UserMonthYear"], inplace=True)

svm_pred.to_csv("svmPred.csv", index = False)